# Day 051 | Classification Metrics (ROC, AUC, F1)

## 100 Days of ML for Geotechnical Engineering
**Phase:** Classical Machine Learning

**Author:** Ripon Chandra Malo  
**Date:** YYYY-MM-DD  

---

## What We Will Learn Today

| Concept | What It Means | Geotechnical Example |
|---------|--------------|---------------------|
| Confusion Matrix | TP, FP, TN, FN counts | Correctly identifying liquefiable sites |
| Precision | Of predicted positives, how many correct | "Of sites flagged risky, how many actually are?" |
| Recall (Sensitivity) | Of actual positives, how many caught | "Of truly risky sites, how many did we catch?" |
| F1-Score | Harmonic mean of precision and recall | Balanced metric when both matter |
| ROC Curve / AUC | TPR vs FPR tradeoff | Overall model discrimination ability |
| Precision-Recall Curve | Better for imbalanced datasets | When liquefiable sites are rare |
| Threshold Tuning | Adjust decision boundary for safety | Lower threshold to catch ALL at-risk sites |

---

## Our Geotechnical Problem

We classify soil sites as **liquefiable (1)** or **non-liquefiable (0)** based on in-situ data.  
**Missing a liquefiable site is far worse than a false alarm** — so we prefer **high recall**.

| Outcome | Meaning | Consequence |
|---------|---------|-------------|
| True Positive (TP) | Correctly flagged as liquefiable | Safe — we mitigate the risk |
| False Negative (FN) | Missed a liquefiable site | **Dangerous** — no mitigation done |
| False Positive (FP) | Flagged safe site as risky | Costly — unnecessary remediation |
| True Negative (TN) | Correctly identified safe site | Good — no action needed |

---

## Part 1: Generate Synthetic Liquefaction Data

In [ ]:
# ─────────────────────────────────────────────────────
# IMPORTS AND DATA GENERATION
# ─────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
n_samples = 600

# Features
depth = np.random.uniform(1.0, 20.0, n_samples)
n60 = np.random.uniform(2, 50, n_samples)
fines_content = np.random.uniform(0, 70, n_samples)
water_table = np.random.uniform(0.5, 8.0, n_samples)
pga = np.random.uniform(0.05, 0.5, n_samples)          # peak ground acceleration (g)
csr = 0.65 * pga * (depth * 18) / (depth * 18 * 0.6) * 0.9  # simplified CSR

# Liquefaction susceptibility (nonlinear boundary)
crr = 0.02 * n60**0.8 + 0.005 * fines_content + 0.01
factor_of_safety = crr / (csr + 1e-6)
prob_liq = 1 / (1 + np.exp(5 * (factor_of_safety - 1.0)))
prob_liq += np.random.normal(0, 0.08, n_samples)
liquefied = (prob_liq > 0.5).astype(int)

df = pd.DataFrame({
    'depth_m': depth,
    'N60': n60,
    'fines_pct': fines_content,
    'water_table_m': water_table,
    'PGA_g': pga,
    'CSR': csr,
    'liquefied': liquefied
})

df.to_csv('liquefaction_data.csv', index=False)
print(f"Dataset shape: {df.shape}")
print(f"\nClass distribution:")
print(df['liquefied'].value_counts())
print(f"\nLiquefaction rate: {df['liquefied'].mean()*100:.1f}%")

---

## Part 2: Train a Classification Model

In [ ]:
# ─────────────────────────────────────────────────────
# TRAIN/TEST SPLIT AND MODEL TRAINING
# ─────────────────────────────────────────────────────

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

features = ['depth_m', 'N60', 'fines_pct', 'water_table_m', 'PGA_g', 'CSR']
X = df[features]
y = df['liquefied']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

model = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)
model.fit(X_train_s, y_train)

y_pred = model.predict(X_test_s)
y_prob = model.predict_proba(X_test_s)[:, 1]  # probability of class 1

print(f"Training: {X_train.shape[0]} samples")
print(f"Test:     {X_test.shape[0]} samples")
print(f"Test class balance: {y_test.value_counts().to_dict()}")

---

## Part 3: Confusion Matrix — The Foundation

In [ ]:
# ─────────────────────────────────────────────────────
# CONFUSION MATRIX — The foundation of all metrics
# ─────────────────────────────────────────────────────

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Visual confusion matrix
ConfusionMatrixDisplay(cm, display_labels=['Safe (0)', 'Liquefiable (1)']).plot(
    ax=axes[0], cmap='Blues', colorbar=False
)
axes[0].set_title('Confusion Matrix', fontsize=13)

# Annotated explanation
axes[1].axis('off')
table_data = [
    ['', 'Predicted Safe', 'Predicted Liq.'],
    ['Actually Safe', f'TN = {tn}\n(Correct)', f'FP = {fp}\n(False alarm)'],
    ['Actually Liq.', f'FN = {fn}\n(MISSED!)', f'TP = {tp}\n(Caught)']
]
colors = [
    ['white', 'white', 'white'],
    ['white', '#c8e6c9', '#fff9c4'],
    ['white', '#ffcdd2', '#c8e6c9']
]
table = axes[1].table(cellText=table_data, cellColours=colors,
                       loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.3, 2.0)
axes[1].set_title('What Each Cell Means', fontsize=13)

plt.suptitle('Confusion Matrix — Liquefaction Classification',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"True Positives (caught):  {tp}")
print(f"True Negatives (correct): {tn}")
print(f"False Positives (alarm):  {fp}")
print(f"False Negatives (MISSED): {fn}  <-- most dangerous error")

---

## Part 4: Precision, Recall, F1-Score, and Classification Report

In [ ]:
# ─────────────────────────────────────────────────────
# PRECISION, RECALL, F1-SCORE
# ─────────────────────────────────────────────────────

from sklearn.metrics import (precision_score, recall_score, f1_score,
                             accuracy_score, classification_report)

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)

print("Classification Metrics:")
print(f"  Accuracy:  {accuracy:.4f}  <- % of all predictions correct")
print(f"  Precision: {precision:.4f}  <- of predicted liquefiable, {precision*100:.1f}% truly are")
print(f"  Recall:    {recall:.4f}  <- of actual liquefiable sites, caught {recall*100:.1f}%")
print(f"  F1-Score:  {f1:.4f}  <- harmonic mean of precision and recall")
print()
print("For SAFETY: Recall is the critical metric!")
print(f"  We caught {tp} of {tp+fn} liquefiable sites ({recall*100:.1f}%).")
print(f"  We MISSED {fn} liquefiable sites — this is the dangerous outcome.")
print()
print("Full Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Safe', 'Liquefiable']))

---

## Part 5: ROC Curve and AUC

In [ ]:
# ─────────────────────────────────────────────────────
# ROC CURVE — True Positive Rate vs False Positive Rate
# ─────────────────────────────────────────────────────

from sklearn.metrics import roc_curve, roc_auc_score

fpr, tpr, thresholds_roc = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ROC Curve
axes[0].plot(fpr, tpr, 'b-', linewidth=2, label=f'RF Model (AUC = {auc:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC = 0.500)')
axes[0].fill_between(fpr, tpr, alpha=0.15, color='blue')
axes[0].set_xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
axes[0].set_ylabel('True Positive Rate (Recall)', fontsize=12)
axes[0].set_title('ROC Curve', fontsize=13)
axes[0].legend(fontsize=11, loc='lower right')
axes[0].set_xlim([-0.02, 1.02])
axes[0].set_ylim([-0.02, 1.02])

# AUC interpretation guide
axes[1].axis('off')
interp = [
    ['AUC Range', 'Interpretation', 'Geotech Action'],
    ['0.90 - 1.00', 'Excellent', 'High confidence in classification'],
    ['0.80 - 0.90', 'Good', 'Useful, verify borderline cases'],
    ['0.70 - 0.80', 'Fair', 'Need additional testing data'],
    ['0.50 - 0.70', 'Poor', 'Model barely better than guessing'],
    ['< 0.50', 'Fail', 'Model is worse than random']
]
table = axes[1].table(cellText=interp, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.5, 1.8)
for j in range(3):
    table[0, j].set_facecolor('#2c3e50')
    table[0, j].set_text_props(color='white', fontweight='bold')
axes[1].set_title(f'Our Model AUC = {auc:.3f}', fontsize=13, fontweight='bold')

plt.suptitle('ROC Curve — Liquefaction Classification',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## Part 6: Precision-Recall Curve

In [ ]:
# ─────────────────────────────────────────────────────
# PRECISION-RECALL CURVE — Better for imbalanced data
# ─────────────────────────────────────────────────────

from sklearn.metrics import precision_recall_curve, average_precision_score

prec_arr, rec_arr, thresholds_pr = precision_recall_curve(y_test, y_prob)
avg_prec = average_precision_score(y_test, y_prob)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(rec_arr, prec_arr, 'b-', linewidth=2,
        label=f'RF Model (AP = {avg_prec:.3f})')
ax.fill_between(rec_arr, prec_arr, alpha=0.15, color='blue')
ax.axhline(y=y_test.mean(), color='red', linestyle='--',
           label=f'No-skill baseline ({y_test.mean():.2f})')

# Mark high-recall zone
ax.axvspan(0.9, 1.0, alpha=0.1, color='green', label='High recall zone (>0.9)')

ax.set_xlabel('Recall (Sensitivity)', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curve\nFor engineering safety, we want HIGH RECALL',
             fontsize=13)
ax.legend(fontsize=11)
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])

plt.tight_layout()
plt.show()

print("For liquefaction: high recall = catch all at-risk sites.")
print("PR curve is more informative than ROC when classes are imbalanced.")

---

## Part 7: Threshold Tuning for Engineering Safety

By default, sklearn uses a threshold of **0.5**: if P(liquefiable) > 0.5, predict liquefiable.  
But for **safety**, we should lower the threshold — flag a site as risky even at 0.3 probability.

In [ ]:
# ─────────────────────────────────────────────────────
# THRESHOLD TUNING — Adjust for safety
# ─────────────────────────────────────────────────────

thresholds_to_try = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]

results = []
for t in thresholds_to_try:
    y_pred_t = (y_prob >= t).astype(int)
    cm_t = confusion_matrix(y_test, y_pred_t)
    tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
    results.append({
        'Threshold': t,
        'Precision': precision_score(y_test, y_pred_t, zero_division=0),
        'Recall': recall_score(y_test, y_pred_t, zero_division=0),
        'F1': f1_score(y_test, y_pred_t, zero_division=0),
        'Accuracy': accuracy_score(y_test, y_pred_t),
        'FN (Missed)': fn_t,
        'FP (Alarms)': fp_t
    })

thresh_df = pd.DataFrame(results).round(4)
print("\nThreshold Tuning Results:")
print("="*80)
print(thresh_df.to_string(index=False))
print("="*80)
print("\nEngineering priority: minimize FN (missed liquefiable sites).")
print("Lower threshold = higher recall = fewer missed sites, more false alarms.")

In [ ]:
# ─────────────────────────────────────────────────────
# VISUALIZE THRESHOLD TRADEOFF
# ─────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Precision vs Recall vs Threshold
axes[0].plot(thresh_df['Threshold'], thresh_df['Precision'], 'b-o',
             linewidth=2, label='Precision', markersize=8)
axes[0].plot(thresh_df['Threshold'], thresh_df['Recall'], 'r-s',
             linewidth=2, label='Recall', markersize=8)
axes[0].plot(thresh_df['Threshold'], thresh_df['F1'], 'g-^',
             linewidth=2, label='F1-Score', markersize=8)
axes[0].axvline(x=0.5, color='gray', linestyle='--', alpha=0.5, label='Default (0.5)')
axes[0].set_xlabel('Decision Threshold', fontsize=12)
axes[0].set_ylabel('Score', fontsize=12)
axes[0].set_title('Metrics vs Threshold', fontsize=13)
axes[0].legend(fontsize=10)

# Right: FN and FP vs Threshold
axes[1].bar(np.array(thresholds_to_try) - 0.02, thresh_df['FN (Missed)'],
            width=0.04, color='#e74c3c', label='FN (Missed — DANGEROUS)', edgecolor='black')
axes[1].bar(np.array(thresholds_to_try) + 0.02, thresh_df['FP (Alarms)'],
            width=0.04, color='#f39c12', label='FP (False alarms — costly)', edgecolor='black')
axes[1].set_xlabel('Decision Threshold', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title('Error Types vs Threshold', fontsize=13)
axes[1].legend(fontsize=10)

plt.suptitle('Threshold Tuning: Safety vs Cost Tradeoff',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Key insight: Lowering threshold from 0.5 to 0.3 may catch more")
print("liquefiable sites at the cost of more false alarms (extra testing).")

---

## Part 8: Probability Distribution by Class

In [ ]:
# ─────────────────────────────────────────────────────
# PROBABILITY DISTRIBUTIONS — Understanding model confidence
# ─────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(10, 6))

safe_probs = y_prob[y_test == 0]
liq_probs = y_prob[y_test == 1]

ax.hist(safe_probs, bins=25, alpha=0.6, color='#2ecc71', label='Actually Safe',
        edgecolor='black', density=True)
ax.hist(liq_probs, bins=25, alpha=0.6, color='#e74c3c', label='Actually Liquefiable',
        edgecolor='black', density=True)

# Show threshold lines
for t, ls in [(0.3, '-.'), (0.5, '--')]:
    ax.axvline(x=t, color='black', linestyle=ls, linewidth=2,
               label=f'Threshold = {t}')

ax.set_xlabel('Predicted Probability of Liquefaction', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Predicted Probability Distribution by True Class',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print("Ideally: Safe sites cluster near 0, liquefiable sites cluster near 1.")
print("The overlap region is where the model is uncertain — threshold choice matters there.")

---

## Practice Exercises

### Exercise 1: Optimize for Recall
Find the threshold that achieves at least 95% recall. What is the corresponding precision and F1-score?

### Exercise 2: Compare Classifiers
Train a Logistic Regression and an SVM on the same data. Plot all three ROC curves on one figure and compare AUC values.

### Exercise 3: Class Imbalance
Resample the data so only 10% of sites are liquefiable. Retrain and compare ROC vs Precision-Recall curves. Which is more informative?

In [ ]:
# EXERCISE 1: Find threshold for 95% recall
# for t in np.arange(0.05, 0.95, 0.05):
#     y_pred_t = (y_prob >= t).astype(int)
#     rec = recall_score(y_test, y_pred_t)
#     if rec >= 0.95:
#         prec = precision_score(y_test, y_pred_t)
#         f1_t = f1_score(y_test, y_pred_t)
#         print(f"Threshold={t:.2f}, Recall={rec:.3f}, "
#               f"Precision={prec:.3f}, F1={f1_t:.3f}")
#         break

In [ ]:
# EXERCISE 2: Compare multiple classifiers
# from sklearn.linear_model import LogisticRegression
# from sklearn.svm import SVC
# lr = LogisticRegression(max_iter=1000).fit(X_train_s, y_train)
# svm = SVC(probability=True).fit(X_train_s, y_train)
# # Get probabilities, compute ROC for each, plot all on one figure

---

## Key Takeaways

1. **Accuracy alone is misleading** — especially with imbalanced classes. A model predicting "safe" for everything gets high accuracy if most sites are safe, but catches zero risks.

2. **Recall** is the critical metric for geotechnical safety. Missing a liquefiable site (false negative) can lead to structural failure. Prefer models with high recall even at the cost of more false alarms.

3. **ROC-AUC** measures overall model quality across all thresholds. **Precision-Recall curves** are more informative when classes are imbalanced.

4. **Threshold tuning** lets you adjust the precision-recall tradeoff. For safety applications, lower the threshold below 0.5 to catch more at-risk sites.

5. The **confusion matrix** is the foundation — every other metric is derived from TP, FP, TN, FN. Always start here.

---

## How This Connects to ML

| Today's Concept | ML Equivalent |
|----------------|---------------|
| Confusion matrix | Foundation for all classification metrics |
| Precision / Recall | Tunable tradeoff based on domain needs |
| F1-Score | Balanced metric when both precision and recall matter |
| ROC / AUC | Model comparison metric, threshold-independent |
| Threshold tuning | Adjusting decision boundary for business/safety needs |
| Class imbalance | Why accuracy fails — need recall or PR-AUC instead |
| Classification report | sklearn one-line summary of all metrics |

---

*Day 051 of 100 — In safety-critical engineering, recall saves lives!*